# Monte Carlo Sampling with Richards-Wolf Intensity Patterns

This notebook demonstrates how to use Richards-Wolf intensity distributions as sampling patterns for Monte Carlo photon simulations.

We'll show how to:
1. Compute Richards-Wolf intensity pattern
2. Use it for rejection sampling of photon positions
3. Compare with scalar (Airy) pattern sampling

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import j1
from monte_carlo import RichardsWolfSimulator, ApertureSimulator
from monte_carlo import metrics

sns.set_theme(style="whitegrid", font_scale=1.5)

import os
output_dir = '../data/richards_wolf'
os.makedirs(output_dir, exist_ok=True)

## Setup: High-NA System

In [ ]:
# System parameters
wavelength = 0.532  # microns
NA = 0.9  # High NA where vector effects matter
n_medium = 1.0
n_photons = 100000

# Create Richards-Wolf simulator for intensity pattern
rw = RichardsWolfSimulator(
    wavelength=wavelength,
    numerical_aperture=NA,
    n_medium=n_medium,
    polarization='x'
)

print(f"Wavelength: {wavelength} μm")
print(f"NA: {NA}")
print(f"N photons: {n_photons}")
print(f"Airy radius: {rw.airy_radius:.4f} μm")

## Method 1: Rejection Sampling with Richards-Wolf Pattern

Sample photon positions from the focal plane using the Richards-Wolf intensity as probability distribution.

In [ ]:
def sample_from_intensity_pattern(intensity_func, n_samples, r_max, seed=None):
    """
    Sample (x, y) positions using rejection sampling.
    
    Parameters
    ----------
    intensity_func : callable
        Function that takes (x, y) and returns normalized intensity
    n_samples : int
        Number of samples to generate
    r_max : float
        Maximum radial extent for sampling
    seed : int, optional
        Random seed
    
    Returns
    -------
    x, y : np.ndarray
        Sampled positions
    """
    if seed is not None:
        np.random.seed(seed)
    
    accepted_x = []
    accepted_y = []
    
    while len(accepted_x) < n_samples:
        # Generate candidate points in circular region
        n_candidates = n_samples * 2
        r_candidate = r_max * np.sqrt(np.random.random(n_candidates))
        theta_candidate = 2 * np.pi * np.random.random(n_candidates)
        
        x_candidate = r_candidate * np.cos(theta_candidate)
        y_candidate = r_candidate * np.sin(theta_candidate)
        
        # Calculate intensity
        intensity = intensity_func(x_candidate, y_candidate)
        
        # Rejection sampling
        accept_prob = np.random.random(n_candidates)
        accepted_mask = accept_prob < intensity
        
        accepted_x.extend(x_candidate[accepted_mask])
        accepted_y.extend(y_candidate[accepted_mask])
    
    return np.array(accepted_x[:n_samples]), np.array(accepted_y[:n_samples])

print("Sampling photons from Richards-Wolf pattern...")
sampling_radius = 5 * rw.airy_radius

x_rw, y_rw = sample_from_intensity_pattern(
    intensity_func=rw.focal_plane_intensity_pattern,
    n_samples=n_photons,
    r_max=sampling_radius,
    seed=42
)

print(f"Sampled {len(x_rw)} photons")
print(f"Mean r: {np.sqrt(x_rw**2 + y_rw**2).mean():.4f} μm")

## Method 2: Scalar (Airy) Pattern for Comparison

Use the ApertureSimulator with scalar Airy pattern.

In [ ]:
# Create aperture simulator (uses Airy pattern)
# Note: focal_length parameter doesn't affect the intensity pattern, only ray geometry
aperture_sim = ApertureSimulator(
    n_photons=n_photons,
    wavelength=wavelength,
    focal_length=10.0,  # arbitrary
    numerical_aperture=NA,
    n_medium=n_medium,
    random_seed=42
)

print("Sampling photons from scalar Airy pattern...")
x_airy, y_airy = aperture_sim.sample_focal_plane_position()

print(f"Sampled {len(x_airy)} photons")
print(f"Mean r: {np.sqrt(x_airy**2 + y_airy**2).mean():.4f} μm")

## Visualize Sampled Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

plot_range = 2 * rw.airy_radius
bins = 150

# Richards-Wolf sampling
ax = axes[0]
h = ax.hist2d(x_rw, y_rw, bins=bins, 
              range=[[-plot_range, plot_range], [-plot_range, plot_range]],
              cmap='hot')
plt.colorbar(h[3], ax=ax, label='Count')
ax.set_xlabel('x (μm)')
ax.set_ylabel('y (μm)')
ax.set_title('Richards-Wolf Sampling', fontweight='bold')
ax.set_aspect('equal')

# Scalar Airy sampling
ax = axes[1]
h = ax.hist2d(x_airy, y_airy, bins=bins,
              range=[[-plot_range, plot_range], [-plot_range, plot_range]],
              cmap='hot')
plt.colorbar(h[3], ax=ax, label='Count')
ax.set_xlabel('x (μm)')
ax.set_ylabel('y (μm)')
ax.set_title('Scalar (Airy) Sampling', fontweight='bold')
ax.set_aspect('equal')

# Difference
ax = axes[2]
hist_rw, xedges, yedges = np.histogram2d(x_rw, y_rw, bins=bins,
                                          range=[[-plot_range, plot_range], 
                                                 [-plot_range, plot_range]])
hist_airy, _, _ = np.histogram2d(x_airy, y_airy, bins=bins,
                                  range=[[-plot_range, plot_range], 
                                         [-plot_range, plot_range]])

# Normalize
hist_rw_norm = hist_rw / np.max(hist_rw)
hist_airy_norm = hist_airy / np.max(hist_airy)
diff = hist_rw_norm - hist_airy_norm

im = ax.imshow(diff.T, origin='lower', cmap='RdBu_r',
               extent=[-plot_range, plot_range, -plot_range, plot_range],
               vmin=-0.2, vmax=0.2)
plt.colorbar(im, ax=ax, label='Difference')
ax.set_xlabel('x (μm)')
ax.set_ylabel('y (μm)')
ax.set_title('Difference (RW - Airy)', fontweight='bold')
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(f'{output_dir}/mc_sampling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Radial Profile Comparison

In [ ]:
# Create radial histograms
r_bins = np.linspace(0, plot_range, 300)
r_centers = (r_bins[:-1] + r_bins[1:]) / 2
bin_areas = np.pi * (r_bins[1:]**2 - r_bins[:-1]**2)

# Richards-Wolf
r_rw = np.sqrt(x_rw**2 + y_rw**2)
hist_rw, _ = np.histogram(r_rw, bins=r_bins)
intensity_rw = hist_rw / bin_areas
intensity_rw = intensity_rw / np.max(intensity_rw)

# Airy
r_airy = np.sqrt(x_airy**2 + y_airy**2)
hist_airy, _ = np.histogram(r_airy, bins=r_bins)
intensity_airy = hist_airy / bin_areas
intensity_airy = intensity_airy / np.max(intensity_airy)

# Theoretical patterns
x_theory = r_centers
y_theory = np.zeros_like(r_centers)
theory_rw = rw.focal_plane_intensity_pattern(x_theory, y_theory)

k = 2 * np.pi * NA / wavelength
kr = k * r_centers
theory_airy = np.ones_like(kr)
nonzero = kr != 0
theory_airy[nonzero] = (2 * j1(kr[nonzero]) / kr[nonzero])**2

# Plot
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Full comparison
ax = axes[0]
ax.plot(r_centers, theory_rw, linewidth=5, alpha=0.4, color='tab:blue', 
        label='Theory: Richards-Wolf', zorder=3)
ax.plot(r_centers, theory_airy, linewidth=5, alpha=0.4, color='tab:orange',
        label='Theory: Airy', zorder=3)
ax.plot(r_centers, intensity_rw, linewidth=3, linestyle='--', color='tab:blue',
        label='MC: Richards-Wolf')
ax.plot(r_centers, intensity_airy, linewidth=3, linestyle='--', color='tab:orange',
        label='MC: Airy')
ax.axvline(rw.airy_radius, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Radial distance r (μm)')
ax.set_ylabel('Normalized Intensity')
ax.set_title(f'Radial Profiles (NA={NA})', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, plot_range])

# Zoomed
ax = axes[1]
zoom_range = 1.5 * rw.airy_radius
zoom_mask = r_centers < zoom_range
ax.plot(r_centers[zoom_mask], theory_rw[zoom_mask], linewidth=5, alpha=0.4, 
        color='tab:blue', label='Theory: Richards-Wolf', zorder=3)
ax.plot(r_centers[zoom_mask], theory_airy[zoom_mask], linewidth=5, alpha=0.4, 
        color='tab:orange', label='Theory: Airy', zorder=3)
ax.plot(r_centers[zoom_mask], intensity_rw[zoom_mask], linewidth=3, linestyle='--',
        color='tab:blue', label='MC: Richards-Wolf')
ax.plot(r_centers[zoom_mask], intensity_airy[zoom_mask], linewidth=3, linestyle='--',
        color='tab:orange', label='MC: Airy')
ax.axvline(rw.airy_radius, color='gray', linestyle=':', alpha=0.5,
           label=f'Airy radius ({rw.airy_radius:.3f} μm)')
ax.set_xlabel('Radial distance r (μm)')
ax.set_ylabel('Normalized Intensity')
ax.set_title('Radial Profiles (Zoomed)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/mc_radial_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Quantitative Comparison

In [ ]:
# Richards-Wolf vs theory
rmse_rw_theory = metrics.rmse(intensity_rw, theory_rw)
fid_rw_theory = metrics.fidelity(theory_rw, intensity_rw)

# Airy vs theory  
rmse_airy_theory = metrics.rmse(intensity_airy, theory_airy)
fid_airy_theory = metrics.fidelity(theory_airy, intensity_airy)

# RW vs Airy (MC samples)
rmse_rw_airy = metrics.rmse(intensity_rw, intensity_airy)
fid_rw_airy = metrics.fidelity(intensity_airy, intensity_rw)

print("=" * 80)
print(f"MONTE CARLO SAMPLING COMPARISON (NA={NA}, N={n_photons})")
print("=" * 80)
print(f"{'Comparison':<40} {'RMSE':<15} {'Fidelity':<15}")
print("-" * 80)
print(f"{'MC (RW) vs Theory (RW)':<40} {rmse_rw_theory:<15.6f} {fid_rw_theory:<15.6f}")
print(f"{'MC (Airy) vs Theory (Airy)':<40} {rmse_airy_theory:<15.6f} {fid_airy_theory:<15.6f}")
print(f"{'MC (RW) vs MC (Airy)':<40} {rmse_rw_airy:<15.6f} {fid_rw_airy:<15.6f}")
print("=" * 80)
print("\nConclusion:")
print(f"  - Both MC methods accurately sample from their respective theories")
print(f"  - Difference between RW and Airy: RMSE = {rmse_rw_airy:.6f}")
print(f"  - At NA={NA}, vectorial effects are significant")

## Summary

This notebook demonstrated:

1. **Using Richards-Wolf for Monte Carlo sampling**: The `focal_plane_intensity_pattern()` method can be used directly with rejection sampling

2. **High-NA effects**: At NA=0.9, there are measurable differences between vectorial (Richards-Wolf) and scalar (Airy) patterns

3. **Integration with existing code**: The Richards-Wolf simulator is compatible with the existing Monte Carlo framework

### Next Steps

- Create a modified `ApertureSimulator` that uses Richards-Wolf intensity instead of Airy
- Study convergence properties of MC sampling from Richards-Wolf patterns
- Compare computational cost vs accuracy tradeoffs